### Atividade Final
- Aberto: Sunday, 10 Nov 2024, 00:00
- Vencimento: Wednesday, 4 Dec 2024, 23:59

Gere um arquivo CSV a partir de uma planilha, converta o csv para RDF e importe no banco de dados Virtuoso open source (http://200.129.247.238:8890/conductor). Execute 5 consultas sobre o assunto no SPARQL.

Enviar:

1) arquivo no DOC ou PDF com o print da planilha em  CSV

2) Insira o print do arquivo RDF gerado pelo OpenRefine ou Python 

3) Insira o print das consultas realizadas no virtuoso com os resultados das triplas geradas

### 1 - Importar biblioteca e ler a base CSV

In [41]:
import pandas as pd

In [42]:
# Carregar a base de dados dos games
base = "games.csv"
df = pd.read_csv(base, sep=",", nrows=100)

# Visualizar as primeiras 20 linhas
print(df.head(20))

   nome_;genero_;desenvolvedor_;lancamento_;plataformas_
0    Valorant Reborn;MMORPG;Bungie;1986;PlayStation 5   
1   PUBG Origins;Corrida;Blizzard Entertainment;20...   
2   Assassin's Creed Odyssey;Corrida;Epic Games;19...   
3   Among Us Origins;Puzzle;Mojang;2008;Xbox Series X   
4   Overwatch Origins;Esporte;Supercell;1994;Xbox ...   
5   Valorant Chronicles;Estratégia;Epic Games;1997...   
6         Elden Ring Origins;Estratégia;Valve;1993;PC   
7       The Sims Saga;Puzzle;Valve;1987;PlayStation 3   
8   PUBG Reborn;Puzzle;Blizzard Entertainment;2016...   
9   Diablo Odyssey;Sandbox;Mojang;2000;Nintendo Sw...   
10  Stardew Valley Reborn;Sandbox;Activision;1987;...   
11  Valorant Infinity;Ação;FromSoftware;2021;Xbox 360   
12  Red Dead Redemption Origins;RPG;Supercell;2009...   
13  Stardew Valley Saga;FPS;Activision;2023;Xbox S...   
14  Fall Guys Saga;Esporte;Activision;1996;PlaySta...   
15  Diablo Legends;Ação;FromSoftware;1994;PlayStation   
16  Red Dead Redemption Reborn;

### 2 - Imprime informações de toda coluna

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 1 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   nome_;genero_;desenvolvedor_;lancamento_;plataformas_  100 non-null    object
dtypes: object(1)
memory usage: 928.0+ bytes


### 3 - Transformar em RDF Schema
- Formato de dados : Turtle
- Formato de dados : RDF/XML

In [44]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import XSD
import pandas as pd

# Carregar a base de dados
base = "games.csv"
df = pd.read_csv(base, sep=";", nrows=100)

# Criar um gráfico RDF
g = Graph()

# Namespace
namespace = Namespace("http://example.org/games/")

# Sanitizar colunas
sanitized_columns = {col: "".join(e if e.isalnum() else "_" for e in col) for col in df.columns}

# Adicionar dados ao grafo
for index, row in df.iterrows():
    record_uri = URIRef(namespace + f"record/{index}")
    for column, sanitized_column in sanitized_columns.items():
        column_uri = URIRef(namespace + sanitized_column)
        value = row[column]
        if pd.notna(value):
            if column == "plataformas":  # Tratar múltiplas plataformas
                platforms = str(value).split(",")
                for platform in platforms:
                    g.add((record_uri, column_uri, Literal(platform.strip(), datatype=XSD.string)))
            else:
                g.add((record_uri, column_uri, Literal(value, datatype=XSD.string)))


### 4 - Gera as bases em formato Turtle e RDF/XML

In [45]:
# Salvar o gráfico
g.serialize("games.ttl", format="turtle")
g.serialize("games.rdf", format="xml", encoding="ISO-8859-1")
print("Arquivos RDF gerados com sucesso!")

Arquivos RDF gerados com sucesso!


In [46]:
from rdflib import Graph, Namespace

- Visualizar Schema RDF
    https://www.ldf.fi/service/rdf-grapher

----

- Validador Turtle do IDLab
    http://ttl.summerofcode.be/

- Online RDF Validators - Verifica se o RDF está bem formado e pode visualizar o grafo RDF
    https://www.w3.org/RDF/Validator/

----

- RDFShape é um playground para conversão, validação e visualização de dados RDF
    https://rdfshape.weso.es/

    - Análise e visualização de dados                https://rdfshape.weso.es/dataInfo
    - Conversão de dados entre formatos semânticos   https://rdfshape.weso.es/dataConvert
    - Consulta de dados via SPARQL                   https://rdfshape.weso.es/dataQuery


----
- Analisa os dados : https://rdfshape.weso.es/link/17327298766

- Converte Turtle para RDF/XML : https://rdfshape.weso.es/link/17327302251

- Consulta : https://rdfshape.weso.es/dataQuery

---
- Editor de consulta SPARQL
    https://dbpedia.org/sparql